In [ ]:
import os
import json
import pandas as pd
from datetime import datetime

def process_timings(jsonl_path, output_prefix):
    rows = []

    # Read each line from the JSONL file and convert it to a Python dict
    with open(jsonl_path, 'r') as f:
        for line in f:
            try:
                row = json.loads(line)
                rows.append({
                    "type": output_prefix,
                    "start_ms": int(row["start_ms"]),
                    "duration_ms": int(row["duration_ms"])
                })
            except json.JSONDecodeError as e:
                print(f"⚠️ Skipping invalid JSON line: {e}")

    df = pd.DataFrame(rows)
    if df.empty:
        print(f"⚠️ No timing data found in {jsonl_path}. Skipping.")
        return

    base_dir = f"experiments/{output_prefix}"
    # os.makedirs(f"{base_dir}_plots", exist_ok=True)

    # Save raw summary
    df.to_csv(f"{base_dir}_timings_summary.csv", index=False)
    print(f"✅ Saved timing summary to {base_dir}_timings_summary.csv")

    # If timestamps exist, convert and print stats
    if "start_ms" in df.columns and "duration_ms" in df.columns and "type" in df.columns:
        df["start_time"] = pd.to_datetime(df["start_ms"], unit="ms")
        summary_table = df.groupby("type")["duration_ms"].describe()
        print(f"\n📊 Summary Table ({output_prefix}):")
        print(summary_table)
        summary_table.to_csv(f"{base_dir}_timings_stats.csv")
    else:
        print(f"⚠️ Missing required columns in {jsonl_path}. Skipping stats export.")

# --- File mappings: path -> label for output folder
file_scan = {
    "json_files/scan/timings.jsonl": "scan_proof_gen",
    "json_files/scan/features_timings.jsonl": "scan_latency",
    "json_files/scan/verify_timings.jsonl": "scan_verify"
}

for path, label in file_scan.items():
    process_timings(path, label)
